# EEG2GAIT — Hierarchical GCN for EEG-Based Gait Decoding

Full training pipeline on the MoBI dataset. GPU accelerated.


In [ ]:
# Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'scikit-learn'], check=True)
print('Dependencies ready.')


In [ ]:
import os
from pathlib import Path

# Kaggle dataset path
DATA_DIR = Path('/kaggle/input/datasets/wangldan/eeg2gait-fall-prediction-dataset/eeg2gait_data/RepositoryData')
OUTPUT_DIR = Path('/kaggle/working/outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify data is present
if not DATA_DIR.exists():
    print(f'ERROR: {DATA_DIR} does not exist.')
    # Fallback to check what is in /kaggle/input
    print('Contents of /kaggle/input:')
    os.system('ls -R /kaggle/input | head -n 30')
else:
    sessions = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
    print(f'Found {len(sessions)} session folders:')
    for s in sessions[:5]:
        print(f'  {s.name}: eeg={( s/"eeg.txt").exists()}, joints={(s/"joints.txt").exists()}')
    print('...')


## config.py


In [ ]:
%%writefile /kaggle/working/config.py
"""
config.py
---------
Central configuration for the EEG2GAIT pipeline.
All hyper-parameters are defined here so every other module
can import from a single source of truth.
"""

import os
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
ROOT_DIR   = Path("/kaggle")
DATA_DIR   = Path("/kaggle/input/datasets/wangldan/eeg2gait-fall-prediction-dataset/eeg2gait_data/RepositoryData")
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Dataset ──────────────────────────────────────────────────────────────────
SUBJECTS = [f"SL{i:02d}" for i in range(1, 9)]   # SL01 … SL08
SESSIONS = ["T01", "T02", "T03"]

# Raw sampling rate (from data inspection: ~333.3 Hz → 1/0.003)
RAW_FS   = 333.33   # Hz (approximate; actual dt = 0.003 s)

# Target sampling rate after downsampling
TARGET_FS = 100     # Hz

# Band-pass filter  [Hz]
BANDPASS_LO = 0.1
BANDPASS_HI = 48.0

# Number of EEG channels in the file (64 in header, first col is time → 64 data cols)
# 5 channels to drop (EOG / artifact): indices 0-based after removing timestamp
# Typical 64-ch BrainProducts layout: channels 62,63 are EOG; drop any non-brain ch
# Paper says 59 channels are used → drop 5 artifact channels
N_CHANNELS_RAW  = 64
EOG_CHAN_INDICES = [32, 38, 39, 62, 63]   # 0-based in the data matrix (after time col)
N_CHANNELS       = 59                     # channels kept

# EEG electrode 3D positions (approximate from standard 64-ch BrainProducts layout)
# Used to build the graph adjacency matrix (30 mm radius threshold)
# Format: dict mapping 0-based *kept* channel index → (x,y,z) in mm
# Full layout will be built inside dataset.py from the digitizer file if available,
# otherwise a standard 64-ch layout is used.
RADIUS_MM = 30.0    # adjacency radius for GCM

# ── Joints ───────────────────────────────────────────────────────────────────
# joints.txt header says: 6 joints (GHR GKR GAR GHL GKL GAL …)
# The first 6 columns after the timestamp are the *gait* joint angles
JOINT_NAMES = ["GHR", "GKR", "GAR", "GHL", "GKL", "GAL"]
N_JOINTS    = 6     # d_j in the paper

# ── MoBI Data-Split (per session, in minutes) ─────────────────────────────
TRAIN_MIN   = 13.5
VAL_MIN     = 1.5
TEST_MIN    = 5.0

# ── Window / stride ──────────────────────────────────────────────────────────
WINDOW_SECS  = 1.0          # 1-second window
STRIDE_SECS  = 0.1          # 100 ms stride (10-fold overlap)
WINDOW_SAMPS = int(WINDOW_SECS  * TARGET_FS)   # 100 samples
STRIDE_SAMPS = int(STRIDE_SECS * TARGET_FS)    # 10  samples

# ── Model ────────────────────────────────────────────────────────────────────
F_FILTERS     = 25          # LTL temporal filters
LTL_KERNEL    = 10          # kernel size along T axis in LTL conv
HGP_DEPTHS    = [1, 2, 3]  # GCN depths in Hierarchical GCN Pyramid
GSL_DROPOUT   = 0.5
FFN_DROPOUTS  = 0.5
GTL_HEADS     = 4           # multi-head self-attention heads
GTL_DIM       = 200         # feature dim fed into GTL (after FFN)

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE     = 100
LR             = 1e-3
MAX_EPOCHS     = 50
PATIENCE       = 30         # early-stop patience (on val Pearson r)

# ── Loss ─────────────────────────────────────────────────────────────────────
ALPHA   = 0.5    # freq / time weighting
BETA    = 0.1    # reward strength
EPSILON = 1e-8   # numerical stability

# ── Misc ─────────────────────────────────────────────────────────────────────
SEED        = 42
NUM_WORKERS = 0   # set >0 if you have plenty of RAM
DEVICE      = "cuda"   # change to "cuda" / "mps" if available



## dataset.py


In [ ]:
%%writefile /kaggle/working/dataset.py
"""
dataset.py
----------
Custom PyTorch Dataset & DataLoader for the MoBI EEG2GAIT dataset.

File layout expected:
    DATA_DIR/
        SL01-T01/
            eeg.txt     – header: "64 channels\n"
                          data rows: timestamp, ch0, ch1, …, ch63  (tab-separated)
            joints.txt  – header: "6 joints (GHR GKR GAR …)\n"
                          header2: "Joint Factor …\n"
                          data rows: timestamp, j0, j1, j2, j3, j4, j5, (more…)
        SL01-T02/ …

Preprocessing pipeline (per session):
    1. Read raw EEG + joint angles at ~333 Hz
    2. Drop 5 artifact / EOG channels  → 59 channels
    3. Common-Average Reference (CAR)
    4. Band-pass filter  0.1–48 Hz  (scipy butterworth, zero-phase)
    5. Resample to 100 Hz  (scipy.signal.resample_poly)
    6. Align EEG ↔ joints by timestamp (both files share the same timestamps)
    7. Split by time: first 13.5 min → train, next 1.5 min → val, last 5 min → test
    8. Sliding-window extraction: 1-second windows, 100 ms stride
"""

import os
import re
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, filtfilt, resample_poly
from math import gcd

import sys
sys.path.insert(0, '/kaggle/working')
from config import (
    DATA_DIR, SUBJECTS, SESSIONS,
    RAW_FS, TARGET_FS,
    BANDPASS_LO, BANDPASS_HI,
    N_CHANNELS_RAW, EOG_CHAN_INDICES, N_CHANNELS,
    N_JOINTS,
    TRAIN_MIN, VAL_MIN, TEST_MIN,
    WINDOW_SAMPS, STRIDE_SAMPS,
    RADIUS_MM,
)


# ──────────────────────────────────────────────────────────────────────────────
# Helper: I/O
# ──────────────────────────────────────────────────────────────────────────────

def _read_eeg(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """Return (timestamps [N], data [N, 64]) from eeg.txt."""
    with open(path, "r", errors="replace") as f:
        _ = f.readline()  # skip "64 channels" header
        rows = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = line.split("\t")
            try:
                rows.append([float(v) for v in vals if v])
            except ValueError:
                continue
    arr = np.array(rows, dtype=np.float32)
    timestamps = arr[:, 0]          # column 0 is time in seconds
    data       = arr[:, 1:]         # columns 1…64  (64 channels)
    return timestamps, data


def _read_joints(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Return (timestamps [N], joint_angles [N, 6]) from joints.txt.
    The file has two header lines; the joint factor line contains scale factors
    but the raw values in the file are already the actual joint angles in degrees.
    Only the first 6 joints (GHR,GKR,GAR,GHL,GKL,GAL) are used per the paper.
    """
    with open(path, "r", errors="replace") as f:
        h1 = f.readline()   # "6 joints (…)\n"
        h2 = f.readline()   # "Joint Factor …\n"
        rows = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            vals = line.split("\t")
            try:
                rows.append([float(v) for v in vals if v])
            except ValueError:
                continue
    arr = np.array(rows, dtype=np.float32)
    timestamps   = arr[:, 0]
    joint_angles = arr[:, 1:7]      # first 6 joint cols (GHR … GAL)
    return timestamps, joint_angles


# ──────────────────────────────────────────────────────────────────────────────
# Helper: Preprocessing
# ──────────────────────────────────────────────────────────────────────────────

def _drop_eog_channels(data: np.ndarray) -> np.ndarray:
    """Drop EOG/artifact channels; keep 59 of the 64 channels."""
    keep = [i for i in range(N_CHANNELS_RAW) if i not in EOG_CHAN_INDICES]
    return data[:, keep]   # [N, 59]


def _common_average_reference(data: np.ndarray) -> np.ndarray:
    """Subtract the mean across channels at each time point."""
    return data - data.mean(axis=1, keepdims=True)


def _bandpass_filter(data: np.ndarray, fs: float) -> np.ndarray:
    """Zero-phase Butterworth band-pass filter (0.1–48 Hz)."""
    nyq  = fs / 2.0
    lo   = BANDPASS_LO / nyq
    hi   = min(BANDPASS_HI / nyq, 0.999)   # must be < 1
    b, a = butter(4, [lo, hi], btype="band")
    # filtfilt expects (samples,) per channel
    filtered = np.empty_like(data)
    for ch in range(data.shape[1]):
        filtered[:, ch] = filtfilt(b, a, data[:, ch])
    return filtered


def _resample(data: np.ndarray, fs_in: float, fs_out: float) -> np.ndarray:
    """
    Resample from fs_in → fs_out using polyphase method.
    Works channel-by-channel.
    """
    # Build up/down ratio via GCD reduction
    fs_in_int  = int(round(fs_in  * 3))   # × 3 to avoid fractional FS (333.33 Hz → 1000)
    fs_out_int = int(round(fs_out * 3))
    g          = gcd(fs_in_int, fs_out_int)
    up, down   = fs_out_int // g, fs_in_int // g

    resampled = np.empty((round(data.shape[0] * up / down), data.shape[1]), dtype=np.float32)
    for ch in range(data.shape[1]):
        resampled[:, ch] = resample_poly(data[:, ch], up, down).astype(np.float32)
    return resampled


def _preprocess_session(eeg_raw: np.ndarray, joints_raw: np.ndarray,
                         fs: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    Full preprocessing pipeline applied to one session.

    Args:
        eeg_raw:    [N, 64]
        joints_raw: [N, 6]
        fs:         raw sampling frequency

    Returns:
        eeg_pp:    [M, 59]   at TARGET_FS
        joints_pp: [M, 6]    at TARGET_FS
    """
    # 1. Drop artifact channels
    eeg = _drop_eog_channels(eeg_raw)       # [N, 59]

    # 2. Common-Average Reference
    eeg = _common_average_reference(eeg)

    # 3. Band-pass filter
    eeg = _bandpass_filter(eeg, fs)

    # 4. Resample EEG
    eeg = _resample(eeg, fs, TARGET_FS)     # [M, 59]

    # 5. Resample joints
    joints = _resample(joints_raw, fs, TARGET_FS)   # [M, 6]

    # Trim to same length (resampling may differ by ±1)
    n = min(len(eeg), len(joints))
    return eeg[:n], joints[:n]


# ──────────────────────────────────────────────────────────────────────────────
# Helper: Windowing
# ──────────────────────────────────────────────────────────────────────────────

def _extract_windows(eeg: np.ndarray, joints: np.ndarray,
                     window: int = WINDOW_SAMPS,
                     stride: int = STRIDE_SAMPS
                     ) -> Tuple[np.ndarray, np.ndarray]:
    """
    Sliding-window extraction.

    Args:
        eeg:    [T, C]
        joints: [T, J]
    Returns:
        X: [W, C, window]
        y: [W, J]          (label = mean of joint angles in window)
    """
    T = eeg.shape[0]
    indices = list(range(0, T - window + 1, stride))
    X_list, y_list = [], []
    for s in indices:
        e = s + window
        X_list.append(eeg[s:e].T)          # [C, window]
        y_list.append(joints[s:e].mean(0)) # [J]
    if not X_list:
        return np.empty((0, eeg.shape[1], window), dtype=np.float32), \
               np.empty((0, joints.shape[1]), dtype=np.float32)
    return np.stack(X_list).astype(np.float32), \
           np.stack(y_list).astype(np.float32)


# ──────────────────────────────────────────────────────────────────────────────
# Electrode positions (standard BrainProducts 64-ch layout subset, in mm)
# ──────────────────────────────────────────────────────────────────────────────

def _build_standard_positions() -> np.ndarray:
    """
    Returns approximate 3-D positions (mm) for 64 EEG channels in a standard
    BrainProducts layout projected onto a unit sphere of radius 85 mm.
    Only the 59 kept channels (after removing EOG_CHAN_INDICES) are returned.
    Shape: [59, 3]
    """
    # Azimuth / elevation (degrees) for standard 64-ch layout
    # Generated from MNE standard_1020 template reduced to 64 channels.
    # (phi=azimuth, theta=elevation from top, r=85mm)
    az_el_64 = [
        (0,   0),   # Cz
        (180, 18),  # Fz
        (0,  18),   # Pz
        (270, 18),  # C3 (left)
        (90,  18),  # C4 (right)
        (225, 18),  # F3
        (135, 18),  # F4  (approx)
        (315, 18),  # P3
        (45,  18),  # P4
        (180, 36),  # Fpz
        (0,   36),  # Oz
        (270, 36),  # T7
        (90,  36),  # T8
        (225, 36),  # F7
        (135, 36),  # F8
        (315, 36),  # P7
        (45,  36),  # P8
        (247, 28),  # FC5
        (113, 28),  # FC6
        (203, 28),  # FC1
        (157, 28),  # FC2
        (293, 28),  # CP5
        (67,  28),  # CP6
        (247, 28),  # FT9 (approx)
        (113, 28),  # FT10
        (180, 52),  # AF7
        (0,   52),  # O1  (approx)
        (270, 52),  # TP7
        (90,  52),  # TP8
        (225, 52),  # F5
        (135, 52),  # F6
        (315, 52),  # P5
        (45,  52),  # P6
        # fill remaining 31 with evenly spaced positions
        *[(i * (360/31), 72) for i in range(31)],
    ]
    r = 85.0   # sphere radius in mm
    positions = np.zeros((64, 3), dtype=np.float32)
    for i, (az, el) in enumerate(az_el_64):
        az_r = np.radians(az)
        el_r = np.radians(el)
        positions[i, 0] = r * np.sin(el_r) * np.cos(az_r)
        positions[i, 1] = r * np.sin(el_r) * np.sin(az_r)
        positions[i, 2] = r * np.cos(el_r)
    # Keep only non-EOG channels
    keep = [i for i in range(64) if i not in EOG_CHAN_INDICES]
    return positions[keep]   # [59, 3]


def build_adjacency_matrix(positions: Optional[np.ndarray] = None,
                            radius: float = RADIUS_MM) -> np.ndarray:
    """
    Build a binary adjacency matrix A ∈ {0,1}^{C×C} where A_ij=1 if the
    Euclidean distance between electrode i and j is ≤ radius (mm).
    Self-loops are included (A_ii = 1).

    Args:
        positions: [C, 3] electrode coordinates in mm.  If None, the
                   standard layout is used.
        radius:    connectivity radius in mm.

    Returns:
        A: [C, C] float32 array.
    """
    if positions is None:
        positions = _build_standard_positions()   # [59, 3]
    C = positions.shape[0]
    A = np.zeros((C, C), dtype=np.float32)
    for i in range(C):
        for j in range(C):
            dist = np.linalg.norm(positions[i] - positions[j])
            if dist <= radius:
                A[i, j] = 1.0
    # Self-loops
    np.fill_diagonal(A, 1.0)
    return A


# ──────────────────────────────────────────────────────────────────────────────
# PyTorch Dataset
# ──────────────────────────────────────────────────────────────────────────────

class MoBISessionDataset(Dataset):
    """
    Dataset for a single (subject, session, split) tuple.

    Each item: (X, y)
        X: torch.FloatTensor [C, T]   — preprocessed EEG window
        y: torch.FloatTensor [J]      — target joint angles
    """

    def __init__(self,
                 subject: str,
                 session: str,
                 split: str,            # "train" | "val" | "test"
                 data_dir: Path = DATA_DIR,
                 verbose: bool = True):
        assert split in ("train", "val", "test")
        self.subject = subject
        self.session = session
        self.split   = split

        folder = data_dir / f"{subject}-{session}"
        if not folder.exists():
            raise FileNotFoundError(f"Session folder not found: {folder}")

        if verbose:
            print(f"  Loading {subject}-{session} [{split}] …", flush=True)

        # ── Load raw data ──────────────────────────────────────────────────
        ts_eeg,    eeg_raw    = _read_eeg   (folder / "eeg.txt")
        ts_joints, joints_raw = _read_joints(folder / "joints.txt")

        # Infer actual sampling rate from timestamps
        dt = np.median(np.diff(ts_eeg[:500]))
        fs = 1.0 / dt

        # ── Preprocess ────────────────────────────────────────────────────
        eeg_pp, joints_pp = _preprocess_session(eeg_raw, joints_raw, fs)

        # ── Split by time ─────────────────────────────────────────────────
        total_samps = len(eeg_pp)
        train_samps = int(TRAIN_MIN * 60 * TARGET_FS)
        val_samps   = int(VAL_MIN   * 60 * TARGET_FS)
        # test uses remainder (capped at TEST_MIN)
        test_end    = min(total_samps,
                          train_samps + val_samps + int(TEST_MIN * 60 * TARGET_FS))

        if split == "train":
            eeg_split    = eeg_pp   [:train_samps]
            joints_split = joints_pp[:train_samps]
        elif split == "val":
            s = train_samps
            e = train_samps + val_samps
            eeg_split    = eeg_pp   [s:e]
            joints_split = joints_pp[s:e]
        else:   # test
            s = train_samps + val_samps
            eeg_split    = eeg_pp   [s:test_end]
            joints_split = joints_pp[s:test_end]

        # ── Sliding windows ───────────────────────────────────────────────
        self.X, self.y = _extract_windows(eeg_split, joints_split)

        if verbose:
            print(f"    → {len(self.X)} windows, "
                  f"EEG: {eeg_split.shape}, joints: {joints_split.shape}", flush=True)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


class MoBIDataset(Dataset):
    """
    Aggregates data across all (subject, session) pairs for a given split.
    """

    def __init__(self,
                 subjects: List[str] = SUBJECTS,
                 sessions: List[str] = SESSIONS,
                 split: str = "train",
                 data_dir: Path = DATA_DIR,
                 verbose: bool = True):
        self.datasets: List[MoBISessionDataset] = []
        for subj in subjects:
            for sess in sessions:
                folder = data_dir / f"{subj}-{sess}"
                if not folder.exists():
                    if verbose:
                        print(f"  Skipping missing: {subj}-{sess}")
                    continue
                try:
                    ds = MoBISessionDataset(subj, sess, split, data_dir, verbose)
                    if len(ds) > 0:
                        self.datasets.append(ds)
                except Exception as exc:
                    print(f"  Error loading {subj}-{sess}: {exc}")

        # Pre-concatenate for fast indexing
        if self.datasets:
            self.X = np.concatenate([d.X for d in self.datasets], axis=0)
            self.y = np.concatenate([d.y for d in self.datasets], axis=0)
        else:
            self.X = np.empty((0, N_CHANNELS, WINDOW_SAMPS), dtype=np.float32)
            self.y = np.empty((0, N_JOINTS),                 dtype=np.float32)

        if verbose:
            print(f"\n[MoBIDataset-{split}] Total windows: {len(self.X)}")

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx: int):
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])


# ──────────────────────────────────────────────────────────────────────────────
# DataLoader factory
# ──────────────────────────────────────────────────────────────────────────────

def get_dataloaders(subjects: List[str] = SUBJECTS,
                    sessions: List[str] = SESSIONS,
                    batch_size: int      = 100,
                    num_workers: int     = 0,
                    data_dir: Path       = DATA_DIR,
                    verbose: bool        = True
                    ) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Build train / val / test DataLoaders for the given subject / session list.
    """
    print("Building Train dataset …")
    train_ds = MoBIDataset(subjects, sessions, "train", data_dir, verbose)
    print("Building Val   dataset …")
    val_ds   = MoBIDataset(subjects, sessions, "val",   data_dir, verbose)
    print("Building Test  dataset …")
    test_ds  = MoBIDataset(subjects, sessions, "test",  data_dir, verbose)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=False, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=False)
    test_dl  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=False)

    return train_dl, val_dl, test_dl



## model.py


In [ ]:
%%writefile /kaggle/working/model.py
"""
model.py
--------
EEG2GAIT: Hierarchical Graph Convolutional Network for EEG-Based Gait Decoding.

Architecture (following paper Figure 2 and Section III):

  Input X ∈ R^{B × C × T}
    │
    ├─[LTL] Local Temporal Learner       Conv1D(F=25, k=10) → X_ltl ∈ R^{B×F×C×T}
    │       → reshape to  R^{B×F×C×T}   (actually treated as B×F×C×T)
    │
    ├─[GCM] Graph Construction Module    learnable A ∈ R^{C×C}, normalised Â
    │
    ├─[HGP] Hierarchical GCN Pyramid     3 branches (depth 1,2,3) → concat → H
    │
    ├─[GSL] Global Spatial Learner       residual + depth-wise conv(C,1) + BN + ELU
    │                                    + Dropout + AvgPool(1,3) → T//3
    │
    ├─[FFN] Feature Fusion Layers        3 × (Conv(1,10)+BN+ELU+MaxPool) with 50,100,200
    │                                    filters → T//81
    │
    ├─[GTL] Global Temporal Learner      Multi-head self-attention (residual)
    │
    └─[OUT] Output Layer                 Cat(GTL, input) → Conv(dj, 1, T//81×2) → (B, dj)
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List

import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
from config import (
    N_CHANNELS, WINDOW_SAMPS,
    F_FILTERS, LTL_KERNEL,
    HGP_DEPTHS,
    GSL_DROPOUT, FFN_DROPOUTS,
    GTL_HEADS, GTL_DIM,
    N_JOINTS,
)


# ──────────────────────────────────────────────────────────────────────────────
# Graph Convolution Layer
# ──────────────────────────────────────────────────────────────────────────────

class GraphConvLayer(nn.Module):
    """
    One layer of GCN:  H' = σ( Â H W )
    where  Â = D^{-1/2} A D^{-1/2}  (symmetric normalisation)
           H ∈ R^{B × F × C × T}   treated as a graph signal with C nodes
           W ∈ R^{F × F}            learnable weight matrix per feature-map

    For efficiency we apply W as a 1×1 conv over the feature dimension.
    """

    def __init__(self, in_features: int, out_features: int, dropout: float = 0.0):
        super().__init__()
        self.fc      = nn.Linear(in_features, out_features, bias=False)
        self.bn      = nn.BatchNorm1d(out_features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, H: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        """
        H     : [B, C, F]   node features (batch, nodes, feat)
        A_hat : [C, C]      normalised adjacency
        Returns [B, C, F']
        """
        # Graph diffusion: H_out[b,i] = Σ_j A_hat[i,j] * H[b,j]
        # bmm: [B, C, C] × [B, C, F] → but A_hat is shared → use einsum
        AH = torch.einsum("ij,bjf->bif", A_hat, H)  # [B, C, F]
        out = self.fc(AH)                             # [B, C, F']
        # BN over the feature dim (need to permute)
        B, C, Fp = out.shape
        out = self.bn(out.view(B * C, Fp)).view(B, C, Fp)
        out = F.elu(out)
        return self.dropout(out)


# ──────────────────────────────────────────────────────────────────────────────
# Graph Construction Module (GCM)
# ──────────────────────────────────────────────────────────────────────────────

class GraphConstructionModule(nn.Module):
    """
    Maintains one *learnable* adjacency matrix A ∈ R^{C×C}.
    Initialised from the distance-based binary matrix.
    Computes  Â = D^{-1/2} A D^{-1/2}  dynamically.
    """

    def __init__(self, A_init: torch.Tensor):
        """
        A_init: [C, C]  binary adjacency matrix (from distance threshold)
        """
        super().__init__()
        self.A = nn.Parameter(A_init.clone().float())

    def forward(self) -> torch.Tensor:
        """Returns normalised  Â [C, C]."""
        A = F.relu(self.A)          # keep non-negative
        # Degree vector
        D = A.sum(dim=1)            # [C]
        # Avoid div-by-zero for isolated nodes
        D_inv_sqrt = torch.where(D > 0,
                                 torch.pow(D + 1e-8, -0.5),
                                 torch.zeros_like(D))
        D_mat = torch.diag(D_inv_sqrt)  # [C, C]
        A_hat = D_mat @ A @ D_mat       # [C, C]
        return A_hat


# ──────────────────────────────────────────────────────────────────────────────
# Single GCN Branch (one depth)
# ──────────────────────────────────────────────────────────────────────────────

class GCNBranch(nn.Module):
    """
    A stack of `depth` GCN layers.
    Each branch has its OWN independent learnable adjacency matrix.
    """

    def __init__(self, in_feat: int, hidden_feat: int,
                 depth: int, A_init: torch.Tensor, dropout: float = 0.1):
        super().__init__()
        self.gcm    = GraphConstructionModule(A_init)
        layers = []
        for d in range(depth):
            in_f  = in_feat    if d == 0 else hidden_feat
            out_f = hidden_feat
            layers.append(GraphConvLayer(in_f, out_f, dropout))
        self.layers = nn.ModuleList(layers)

    def forward(self, H: torch.Tensor) -> torch.Tensor:
        """H: [B, C, F] → [B, C, F_hidden]"""
        A_hat = self.gcm()
        for layer in self.layers:
            H = layer(H, A_hat)
        return H


# ──────────────────────────────────────────────────────────────────────────────
# Local Temporal Learner (LTL)
# ──────────────────────────────────────────────────────────────────────────────

class LocalTemporalLearner(nn.Module):
    """
    1D temporal convolution per-channel using groups.
    Input  X : [B, C, T]
    Output   : [B, F, C, T]  (F independent temporal filters per channel)

    Implementation note:
    We reshape to [B*C, 1, T] and apply Conv1d with F filters and kernel_size=k.
    Then reshape back to [B, C, F, T] and permute to [B, F, C, T].
    """

    def __init__(self, n_channels: int = N_CHANNELS,
                 n_filters: int = F_FILTERS,
                 kernel_size: int = LTL_KERNEL):
        super().__init__()
        self.n_channels = n_channels
        self.n_filters  = n_filters
        self.conv = nn.Conv1d(
            in_channels  = 1,
            out_channels = n_filters,
            kernel_size  = kernel_size,
            padding      = kernel_size // 2,
            bias         = False
        )
        self.bn  = nn.BatchNorm2d(n_filters)
        self.act = nn.ELU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, C, T] → [B, F, C, T]"""
        B, C, T = x.shape
        # process each channel independently
        x_flat  = x.reshape(B * C, 1, T)           # [B*C, 1, T]
        out     = self.conv(x_flat)                 # [B*C, F, T']
        T_out   = out.shape[-1]
        out     = out.reshape(B, C, self.n_filters, T_out)  # [B,C,F,T']
        out     = out.permute(0, 2, 1, 3)           # [B, F, C, T']
        out     = self.bn(out)
        out     = self.act(out)
        return out  # [B, F, C, T']  where T' ≈ T (zero-padded)


# ──────────────────────────────────────────────────────────────────────────────
# Hierarchical GCN Pyramid (HGP)
# ──────────────────────────────────────────────────────────────────────────────

class HierarchicalGCNPyramid(nn.Module):
    """
    Multiple GCN branches with different depths (1, 2, 3).
    Input:  [B, F, C, T]   (from LTL)
    Applies GCN independently at each time step → outputs [B, depth_branches×F, C, T]

    For efficiency: treat the temporal dimension as batch items.
    Reshape to [B*T, C, F], pass through each branch, then reshape back.
    """

    def __init__(self, in_feat: int, hidden_feat: int,
                 depths: List[int], A_init: torch.Tensor, dropout: float = 0.1):
        super().__init__()
        self.branches = nn.ModuleList([
            GCNBranch(in_feat, hidden_feat, d, A_init, dropout)
            for d in depths
        ])
        self.n_branches = len(depths)
        self.hidden_feat = hidden_feat

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: [B, F, C, T]
        Returns: [B, n_branches*F, C, T]   (concat across branches)
        """
        B, F, C, T = x.shape
        # reshape: treat each (b, t) as a graph-batch item
        x_perm = x.permute(0, 3, 2, 1)     # [B, T, C, F]
        x_flat = x_perm.reshape(B * T, C, F)  # [B*T, C, F]

        branch_outs = []
        for branch in self.branches:
            out = branch(x_flat)             # [B*T, C, F_hidden]
            out = out.reshape(B, T, C, self.hidden_feat)   # [B,T,C,F_h]
            out = out.permute(0, 3, 2, 1)   # [B, F_h, C, T]
            branch_outs.append(out)

        return torch.cat(branch_outs, dim=1)  # [B, n_branches*F_h, C, T]


# ──────────────────────────────────────────────────────────────────────────────
# Global Spatial Learner (GSL)
# ──────────────────────────────────────────────────────────────────────────────

class GlobalSpatialLearner(nn.Module):
    """
    Combines HGP output with original features via residual connection.

    Pipeline per the paper:
      1. Concat [HGP_out, LTL_out] along filter dim  → [B, F_merged, C, T]
      2. Depth-wise spatial collapse: pool across C   → [B, F_merged, 1, T]
      3. Point-wise projection to out_filters         → [B, out_filters, 1, T]
      4. BatchNorm → ELU → Dropout(0.5)
      5. AvgPool(1,3)                                 → [B, out_filters, 1, T//3]
    """

    def __init__(self, n_channels: int,
                 in_filters: int,    # F_hgp (HGP output filters)
                 orig_filters: int,  # F     (LTL output filters)
                 out_filters: int,
                 dropout: float = 0.5):
        super().__init__()
        merged = in_filters + orig_filters

        # Collapse C spatial positions → 1 via mean, then project
        # (equivalent to paper's grouped spatial conv with uniform weights)
        self.spatial_pool = nn.AdaptiveAvgPool2d((1, None))  # [B,F,C,T]→[B,F,1,T]
        self.pointwise    = nn.Conv2d(merged, out_filters, kernel_size=1, bias=False)
        self.bn           = nn.BatchNorm2d(out_filters)
        self.act          = nn.ELU()
        self.drop         = nn.Dropout(dropout)
        self.pool         = nn.AvgPool2d(kernel_size=(1, 3), stride=(1, 3))

    def forward(self, h_hgp: torch.Tensor, x_orig: torch.Tensor) -> torch.Tensor:
        """
        h_hgp : [B, F_hgp, C, T]
        x_orig: [B, F,     C, T]
        Returns [B, out_filters, 1, T//3]
        """
        h = torch.cat([h_hgp, x_orig], dim=1)   # [B, F_merged, C, T]
        h = self.spatial_pool(h)                 # [B, F_merged, 1, T]
        h = self.pointwise(h)                    # [B, out_filters, 1, T]
        h = self.bn(h)
        h = self.act(h)
        h = self.drop(h)
        h = self.pool(h)                         # [B, out_filters, 1, T//3]
        return h


# ──────────────────────────────────────────────────────────────────────────────
# Feature Fusion Network (FFN)  — three sequential blocks
# ──────────────────────────────────────────────────────────────────────────────

class FeatureFusionNetwork(nn.Module):
    """
    Three sequential blocks: Conv(1,10) → BN → ELU → MaxPool(1,3).
    Filter progression: in → 50 → 100 → 200.
    Dropout(0.5) before each block.
    Input/Output shapes:
        in:   [B, in_filters, 1, T_in]
        out:  [B, 200, 1, T_in//27]  (3 poolings of stride 3 → T//3^3 = T//27)
    """

    def __init__(self, in_filters: int, dropout: float = 0.5):
        super().__init__()
        cfg = [(in_filters, 50), (50, 100), (100, 200)]
        self.blocks = nn.ModuleList()
        self.drops  = nn.ModuleList()
        for in_f, out_f in cfg:
            self.blocks.append(nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=(1, 10), padding=(0, 5), bias=False),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3)),
            ))
            self.drops.append(nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, in_filters, 1, T] → [B, 200, 1, T//27]"""
        for drop, block in zip(self.drops, self.blocks):
            x = drop(x)
            x = block(x)
        return x


# ──────────────────────────────────────────────────────────────────────────────
# Global Temporal Learner (GTL) — Multi-head Self-Attention
# ──────────────────────────────────────────────────────────────────────────────

class GlobalTemporalLearner(nn.Module):
    """
    Multi-head self-attention over the temporal dimension with residual connection.

    Input:  [B, F, 1, T_small]
    Treats the T_small time-steps as the sequence length and F as feature dim.
    Output: [B, F, 1, T_small]
    """

    def __init__(self, embed_dim: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim, n_heads,
                                             dropout=dropout, batch_first=True)
        self.norm    = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: [B, F, 1, T] → [B, F, 1, T]"""
        B, F, _, T = x.shape
        # reshape to sequence: [B, T, F]
        x_seq = x.squeeze(2).permute(0, 2, 1)           # [B, T, F]
        attn_out, _ = self.attn(x_seq, x_seq, x_seq)    # [B, T, F]
        attn_out = self.dropout(attn_out)
        x_res    = self.norm(x_seq + attn_out)           # residual + layer-norm
        # reshape back
        out = x_res.permute(0, 2, 1).unsqueeze(2)       # [B, F, 1, T]
        return out


# ──────────────────────────────────────────────────────────────────────────────
# EEG2GAIT — Full Model
# ──────────────────────────────────────────────────────────────────────────────

class EEG2GAIT(nn.Module):
    """
    Full EEG2GAIT model.

    Input:  X ∈ R^{B × C × T}
                C = 59 channels
                T = 100 (1 sec @ 100 Hz)

    Output: ŷ ∈ R^{B × dj}
                dj = 6 joint angles
    """

    def __init__(self,
                 n_channels:   int   = N_CHANNELS,
                 window_samps: int   = WINDOW_SAMPS,
                 n_joints:     int   = N_JOINTS,
                 n_filters:    int   = F_FILTERS,       # F = 25
                 ltl_kernel:   int   = LTL_KERNEL,
                 hgp_depths:   List  = None,
                 hgp_hidden:   int   = 25,               # hidden per GCN branch
                 gtl_heads:    int   = GTL_HEADS,
                 gsl_out:      int   = 50,
                 A_init:       torch.Tensor = None):
        super().__init__()

        if hgp_depths is None:
            hgp_depths = HGP_DEPTHS   # [1, 2, 3]

        self.n_channels   = n_channels
        self.window_samps = window_samps
        self.n_joints     = n_joints

        # ── Compute temporal sizes ────────────────────────────────────────
        # LTL zero-pads to maintain T: T_ltl = T (with even padding)
        T_ltl = window_samps          # ≈ T (see LTL padding logic)
        T_gsl = T_ltl // 3           # after AvgPool(1,3)
        T_ffn = T_gsl // 27          # after 3× MaxPool(1,3)  (3^3=27)
        self.T_ffn = T_ffn

        # ── Modules ──────────────────────────────────────────────────────
        # 1. LTL
        self.ltl = LocalTemporalLearner(n_channels, n_filters, ltl_kernel)

        # 2. HGP
        if A_init is None:
            A_init = torch.ones(n_channels, n_channels)  # fallback
        F_hgp_out = len(hgp_depths) * hgp_hidden
        self.hgp = HierarchicalGCNPyramid(
            in_feat=n_filters, hidden_feat=hgp_hidden,
            depths=hgp_depths, A_init=A_init, dropout=0.1
        )

        # 3. GSL
        self.gsl = GlobalSpatialLearner(
            n_channels  = n_channels,
            in_filters  = F_hgp_out,
            orig_filters = n_filters,
            out_filters  = gsl_out,
            dropout      = GSL_DROPOUT
        )

        # 4. FFN
        self.ffn = FeatureFusionNetwork(in_filters=gsl_out, dropout=FFN_DROPOUTS)
        # Output of FFN: [B, 200, 1, T_ffn]

        # 5. GTL
        self.gtl = GlobalTemporalLearner(embed_dim=200, n_heads=gtl_heads, dropout=0.1)

        # 6. Output layer
        # Concatenate GTL(out) + GTL(in) along T → [B, 200, 1, T_ffn*2]
        # Apply Conv2d(dj, (1, T_ffn*2)) → [B, dj, 1, 1]
        self.out_conv = nn.Conv2d(
            in_channels  = 200,
            out_channels = n_joints,
            kernel_size  = (1, T_ffn * 2),
            bias         = True
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x:  [B, C, T]
        Returns ŷ: [B, dj]
        """
        B, C, T = x.shape

        # ── 1. LTL ────────────────────────────────────────────────────────
        x_ltl = self.ltl(x)        # [B, F, C, T]
        # Ensure T dimension is maintained
        _, F, _, T_ltl = x_ltl.shape

        # ── 2. HGP ────────────────────────────────────────────────────────
        x_hgp = self.hgp(x_ltl)   # [B, n_branches*F_h, C, T_ltl]

        # ── 3. GSL ────────────────────────────────────────────────────────
        x_gsl = self.gsl(x_hgp, x_ltl)   # [B, gsl_out, 1, T_ltl//3]

        # ── 4. FFN ────────────────────────────────────────────────────────
        x_ffn = self.ffn(x_gsl)           # [B, 200, 1, T//81]

        # ── 5. GTL ────────────────────────────────────────────────────────
        x_gtl = self.gtl(x_ffn)           # [B, 200, 1, T//81]

        # ── 6. Output ─────────────────────────────────────────────────────
        # Concatenate along T dimension
        x_cat = torch.cat([x_gtl, x_ffn], dim=3)   # [B, 200, 1, T//81 * 2]

        # Dynamic out_conv kernel if T_ffn changed from init (safety check)
        T_cat = x_cat.shape[3]
        if self.out_conv.kernel_size[1] != T_cat:
            # Re-init output conv with correct size (handles variable-length inputs)
            self.out_conv = nn.Conv2d(
                200, self.n_joints, kernel_size=(1, T_cat), bias=True
            ).to(x.device)

        out = self.out_conv(x_cat)   # [B, dj, 1, 1]
        out = out.squeeze(-1).squeeze(-1)   # [B, dj]
        return out


# ──────────────────────────────────────────────────────────────────────────────
# Factory
# ──────────────────────────────────────────────────────────────────────────────

def build_model(A_init: torch.Tensor = None) -> EEG2GAIT:
    """Construct EEG2GAIT with default configuration."""
    if A_init is None:
        # Identity fallback; caller should pass real adjacency
        A_init = torch.eye(N_CHANNELS)
    return EEG2GAIT(A_init=A_init)



## loss.py


In [ ]:
%%writefile /kaggle/working/loss.py
"""
loss.py
-------
Hybrid Temporal-Spectral Reward (HTSR) Loss.

Paper formulation:
  L_time        = MSE(ŷ, y)
  L_time_reward = L_time + β·log(1 − e^{−L_time} + ε)

  L_freq        = L1(DFT(ŷ), DFT(y))   [over complex magnitudes]
  L_freq_reward = L_freq + β·log(1 − e^{−L_freq} + ε)

  L_total = α·L_freq_reward + (1−α)·L_time_reward

  Default: α=0.5, β=0.1
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
from config import ALPHA, BETA, EPSILON


class HTSRLoss(nn.Module):
    """
    Hybrid Temporal-Spectral Reward Loss.

    Args:
        alpha (float): Weight for frequency-domain loss  (default 0.5)
        beta  (float): Reward strength                   (default 0.1)
        eps   (float): Numerical stability term          (default 1e-8)
    """

    def __init__(self,
                 alpha: float = ALPHA,
                 beta:  float = BETA,
                 eps:   float = EPSILON):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.eps   = eps

    def _reward(self, loss_val: torch.Tensor) -> torch.Tensor:
        """
        Reward term: L + β·log(1 − e^{−L} + ε)
        For large L: log(1 − e^{-L}) → 0  (reward ≈ loss, no penalty)
        For small L: log(1 − e^{-L}) → −∞ (but tempered by β)
        """
        # Clamp to avoid exp underflow/overflow
        L     = loss_val.clamp(min=1e-12)
        inner = 1.0 - torch.exp(-L) + self.eps
        # Guard log argument
        inner = inner.clamp(min=self.eps)
        return L + self.beta * torch.log(inner)

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor
                ) -> torch.Tensor:
        """
        Args:
            y_pred: [B, dj]
            y_true: [B, dj]
        Returns:
            Scalar total loss.
        """
        # ── Time-domain ───────────────────────────────────────────────────
        L_time = F.mse_loss(y_pred, y_true)
        L_time_r = self._reward(L_time)

        # ── Frequency-domain ──────────────────────────────────────────────
        # Compute DFT along joint dimension (or sample-dim if we had sequences)
        # Since predictions are [B, dj] scalars, we apply rfft over the batch
        # dimension (treating batch as a "time" axis) to capture spectral structure
        # across the batch.  Alternatively, if shape is [B, dj], we treat dj as
        # the signal axis.
        # The paper applies DFT to the predicted and true waveforms. Here each
        # sample is already a scalar (mean of window). We use rfft over the dj axis.
        Y_pred_fft = torch.fft.rfft(y_pred, dim=1)   # [B, dj//2+1] complex
        Y_true_fft = torch.fft.rfft(y_true, dim=1)

        # L1 over magnitudes
        L_freq = F.l1_loss(Y_pred_fft.abs(), Y_true_fft.abs())
        L_freq_r = self._reward(L_freq)

        # ── Total ──────────────────────────────────────────────────────────
        L_total = self.alpha * L_freq_r + (1.0 - self.alpha) * L_time_r
        return L_total



## metrics.py


In [ ]:
%%writefile /kaggle/working/metrics.py
"""
metrics.py
----------
Evaluation metrics for EEG2GAIT gait angle prediction.

Metrics (per-joint and averaged):
  - Pearson correlation coefficient (r)
  - R² score (coefficient of determination)
  - Mean Absolute Error (MAE)
"""

import torch
import numpy as np
from typing import Dict, Tuple

import sys
from pathlib import Path
sys.path.insert(0, '/kaggle/working')
from config import JOINT_NAMES, N_JOINTS


def pearson_r(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """
    Pearson correlation coefficient per joint column.

    Args:
        y_pred: [N, J]
        y_true: [N, J]
    Returns:
        r: [J]
    """
    r_vals = []
    for j in range(y_pred.shape[1]):
        p = y_pred[:, j]
        t = y_true[:, j]
        # Avoid NaN for constant predictions
        if np.std(p) < 1e-8 or np.std(t) < 1e-8:
            r_vals.append(0.0)
        else:
            corr = np.corrcoef(p, t)[0, 1]
            r_vals.append(float(corr) if not np.isnan(corr) else 0.0)
    return np.array(r_vals)


def r2_score(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """
    R² (coefficient of determination) per joint.

    Args:
        y_pred: [N, J]
        y_true: [N, J]
    Returns:
        r2: [J]
    """
    ss_res = ((y_true - y_pred) ** 2).sum(axis=0)
    ss_tot = ((y_true - y_true.mean(axis=0)) ** 2).sum(axis=0)
    # Avoid division by zero
    r2 = np.where(ss_tot < 1e-12, 0.0, 1.0 - ss_res / ss_tot)
    return r2


def mae_score(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """
    Mean Absolute Error per joint.

    Args:
        y_pred: [N, J]
        y_true: [N, J]
    Returns:
        mae: [J]
    """
    return np.abs(y_pred - y_true).mean(axis=0)


def compute_metrics(y_pred: np.ndarray,
                    y_true: np.ndarray) -> Dict[str, float]:
    """
    Compute all metrics and return as a flat dict.

    Keys: per-joint (e.g. "r_GHR") and averaged ("r_mean", "r2_mean", "mae_mean").
    """
    r   = pearson_r(y_pred, y_true)
    r2  = r2_score (y_pred, y_true)
    mae = mae_score(y_pred, y_true)

    results: Dict[str, float] = {}
    for j, name in enumerate(JOINT_NAMES):
        results[f"r_{name}"]   = float(r[j])
        results[f"r2_{name}"]  = float(r2[j])
        results[f"mae_{name}"] = float(mae[j])

    results["r_mean"]   = float(r.mean())
    results["r2_mean"]  = float(r2.mean())
    results["mae_mean"] = float(mae.mean())
    return results


def evaluate_loader(model, loader, device: str) -> Dict[str, float]:
    """
    Run inference over the entire DataLoader and compute metrics.

    Returns:
        Dict of metric names → values.
    """
    import torch
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            ŷ = model(X)
            preds.append(ŷ.cpu().numpy())
            targets.append(y.numpy())

    y_pred = np.concatenate(preds,   axis=0)  # [N, J]
    y_true = np.concatenate(targets, axis=0)  # [N, J]
    return compute_metrics(y_pred, y_true)



## train.py


In [ ]:
%%writefile /kaggle/working/train.py
"""
train.py
--------
Training loop for EEG2GAIT.

Features:
  - Adam optimiser (lr=0.001)
  - HTSR custom loss
  - Early stopping: patience=30, monitored metric = mean Pearson r on val set
  - Per-epoch logging: loss + all evaluation metrics
  - Checkpoint saving (best val r model)
  - Final test-set evaluation with per-joint breakdown
"""

import os
import sys
import time
import json
import copy
import random
import argparse
from pathlib import Path

import torch
import numpy as np

# ── Path setup ──────────────────────────────────────────────────────────────
SRC_DIR = Path(__file__).parent
sys.path.insert(0, '/kaggle/working')

from config import (
    DATA_DIR, OUTPUT_DIR, SUBJECTS, SESSIONS,
    BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE,
    SEED, NUM_WORKERS, DEVICE,
    N_CHANNELS, WINDOW_SAMPS, N_JOINTS,
)
from dataset import get_dataloaders, build_adjacency_matrix, _build_standard_positions
from model  import build_model
from loss   import HTSRLoss
from metrics import evaluate_loader, compute_metrics


# ── Reproducibility ─────────────────────────────────────────────────────────

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── Logging helper ───────────────────────────────────────────────────────────

def log(msg: str, log_file=None):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    if log_file:
        log_file.write(line + "\n")
        log_file.flush()


# ── Main training function ──────────────────────────────────────────────────

def train(subjects=None, sessions=None, device_str=None,
          batch_size=BATCH_SIZE, lr=LR, max_epochs=MAX_EPOCHS,
          patience=PATIENCE, output_dir=OUTPUT_DIR):
    """
    Full training pipeline.

    Args:
        subjects:    list of subject IDs (default: all 8)
        sessions:    list of session IDs (default: all 3)
        device_str:  "cpu" | "cuda" | "mps"
        batch_size:  mini-batch size
        lr:          Adam learning rate
        max_epochs:  maximum training epochs
        patience:    early-stopping patience (epochs)
        output_dir:  directory for checkpoints and logs
    """
    set_seed(SEED)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if subjects is None: subjects = SUBJECTS
    if sessions is None: sessions = SESSIONS

    # ── Device ──────────────────────────────────────────────────────────
    if device_str is None:
        device_str = DEVICE
    if device_str == "auto":
        if torch.cuda.is_available():
            device_str = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device_str = "mps"
        else:
            device_str = "cpu"
    device = torch.device(device_str)
    print(f"\n{'='*60}")
    print(f"  EEG2GAIT Training Run")
    print(f"  Device:   {device}")
    print(f"  Subjects: {subjects}")
    print(f"  Sessions: {sessions}")
    print(f"{'='*60}\n")

    log_path = output_dir / "train_log.txt"
    log_file = open(log_path, "w")

    # ── Data ────────────────────────────────────────────────────────────
    log("Loading data …", log_file)
    train_dl, val_dl, test_dl = get_dataloaders(
        subjects=subjects, sessions=sessions,
        batch_size=batch_size, num_workers=NUM_WORKERS,
        verbose=True
    )
    log(f"Train batches: {len(train_dl)} | "
        f"Val batches: {len(val_dl)} | "
        f"Test batches: {len(test_dl)}", log_file)

    # ── Adjacency matrix ─────────────────────────────────────────────────
    log("Building adjacency matrix …", log_file)
    positions = _build_standard_positions()           # [59, 3]
    A_np      = build_adjacency_matrix(positions)     # [59, 59]
    A_init    = torch.from_numpy(A_np)

    # ── Model ────────────────────────────────────────────────────────────
    log("Building model …", log_file)
    model = build_model(A_init=A_init).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"Trainable parameters: {n_params:,}", log_file)

    # ── Optimiser & loss ─────────────────────────────────────────────────
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = HTSRLoss()

    # ── Training state ───────────────────────────────────────────────────
    best_val_r    = -1.0
    best_epoch    = 0
    epochs_no_imp = 0
    best_state    = None
    history       = {"train_loss": [], "val_r": [], "val_r2": [], "val_mae": []}

    # ── Epoch loop ───────────────────────────────────────────────────────
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        # ── Train ────────────────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss  = criterion(y_hat, y)
            loss.backward()
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / max(len(train_dl), 1)

        # ── Validate ─────────────────────────────────────────────────────
        val_metrics = evaluate_loader(model, val_dl, device_str)
        val_r       = val_metrics["r_mean"]
        val_r2      = val_metrics["r2_mean"]
        val_mae     = val_metrics["mae_mean"]

        elapsed = time.time() - t0
        msg = (f"Epoch {epoch:03d}/{max_epochs} | "
               f"Loss: {avg_train_loss:.4f} | "
               f"Val r: {val_r:.4f} | "
               f"Val R²: {val_r2:.4f} | "
               f"Val MAE: {val_mae:.4f} | "
               f"Time: {elapsed:.1f}s")
        log(msg, log_file)

        # Track history
        history["train_loss"].append(avg_train_loss)
        history["val_r"].append(val_r)
        history["val_r2"].append(val_r2)
        history["val_mae"].append(val_mae)

        # ── Early stopping ───────────────────────────────────────────────
        if val_r > best_val_r:
            best_val_r    = val_r
            best_epoch    = epoch
            epochs_no_imp = 0
            best_state    = copy.deepcopy(model.state_dict())
            ckpt_path     = output_dir / "best_model.pt"
            torch.save({
                "epoch":      epoch,
                "state_dict": best_state,
                "val_r":      best_val_r,
                "val_r2":     val_r2,
                "val_mae":    val_mae,
            }, ckpt_path)
            log(f"  ✓ New best model saved (val r={best_val_r:.4f})", log_file)
        else:
            epochs_no_imp += 1

        if epochs_no_imp >= patience:
            log(f"\n  Early stopping triggered after {epoch} epochs "
                f"(best epoch={best_epoch}, best val r={best_val_r:.4f})", log_file)
            break

    # ── Load best model ──────────────────────────────────────────────────
    log(f"\nLoading best model (epoch {best_epoch}) …", log_file)
    if best_state is not None:
        model.load_state_dict(best_state)

    # ── Test evaluation ───────────────────────────────────────────────────
    log("\nEvaluating on TEST set …", log_file)
    test_metrics = evaluate_loader(model, test_dl, device_str)

    log("\n" + "="*60, log_file)
    log("TEST RESULTS", log_file)
    log("="*60, log_file)
    log(f"  Mean Pearson r : {test_metrics['r_mean']:.4f}", log_file)
    log(f"  Mean R²        : {test_metrics['r2_mean']:.4f}", log_file)
    log(f"  Mean MAE       : {test_metrics['mae_mean']:.4f}", log_file)
    log("-"*60, log_file)
    log("  Per-joint breakdown:", log_file)
    from config import JOINT_NAMES
    for name in JOINT_NAMES:
        log(f"    {name:5s}  r={test_metrics[f'r_{name}']:.4f}  "
            f"R²={test_metrics[f'r2_{name}']:.4f}  "
            f"MAE={test_metrics[f'mae_{name}']:.4f}", log_file)
    log("="*60, log_file)

    # ── Save results ─────────────────────────────────────────────────────
    results = {
        "best_epoch":    best_epoch,
        "best_val_r":    best_val_r,
        "test_metrics":  test_metrics,
        "history":       history,
    }
    results_path = output_dir / "results.json"
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    log(f"\nResults saved to: {results_path}", log_file)

    log_file.close()
    return model, results


# ── CLI ──────────────────────────────────────────────────────────────────────

def parse_args():
    p = argparse.ArgumentParser(description="Train EEG2GAIT on the MoBI dataset")
    p.add_argument("--subjects",    nargs="+", default=None,
                   help="Subject IDs to include (e.g. SL01 SL02). Default: all 8.")
    p.add_argument("--sessions",    nargs="+", default=None,
                   help="Session IDs (e.g. T01 T02). Default: all 3.")
    p.add_argument("--device",      default="auto",
                   help="Device: cpu | cuda | mps | auto  (default: auto)")
    p.add_argument("--batch-size",  type=int, default=BATCH_SIZE)
    p.add_argument("--lr",          type=float, default=LR)
    p.add_argument("--max-epochs",  type=int, default=MAX_EPOCHS)
    p.add_argument("--patience",    type=int, default=PATIENCE)
    p.add_argument("--output-dir",  default=str(OUTPUT_DIR))
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train(
        subjects   = args.subjects,
        sessions   = args.sessions,
        device_str = args.device,
        batch_size = args.batch_size,
        lr         = args.lr,
        max_epochs = args.max_epochs,
        patience   = args.patience,
        output_dir = args.output_dir,
    )



## Run Training


In [ ]:
import sys
sys.path.insert(0, '/kaggle/working')

import importlib, config as cfg
from pathlib import Path
cfg.DATA_DIR   = Path('/kaggle/input/datasets/wangldan/eeg2gait-fall-prediction-dataset/eeg2gait_data/RepositoryData')
cfg.OUTPUT_DIR = Path('/kaggle/working/outputs')
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch
print('CUDA available:', torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

from train import train
model, results = train(
    device_str  = device,
    batch_size  = 100,
    max_epochs  = 50,
    patience    = 30,
    output_dir  = '/kaggle/working/outputs',
)


## Results


In [ ]:
import json
with open('/kaggle/working/outputs/results.json') as f:
    res = json.load(f)

tm = res['test_metrics']
print(f"Best epoch : {res['best_epoch']}")
print(f"Best val r : {res['best_val_r']:.4f}")
print()
print('=== TEST SET RESULTS ===')
print(f"  Mean Pearson r : {tm['r_mean']:.4f}")
print(f"  Mean R²        : {tm['r2_mean']:.4f}")
print(f"  Mean MAE       : {tm['mae_mean']:.4f}")
print()
joints = ['GHR','GKR','GAR','GHL','GKL','GAL']
print(f"{'Joint':<6} {'r':>8} {'R²':>8} {'MAE':>8}")
print('-'*34)
for j in joints:
    print(f"{j:<6} {tm[f'r_{j}']:>8.4f} {tm[f'r2_{j}']:>8.4f} {tm[f'mae_{j}']:>8.4f}")
